In [ ]:
import torch
from torch.utils.data import Dataset
from torchvision import datasets
from torchvision.transforms import ToTensor
import matplotlib.pyplot as plt
import json
import requests

# Download all the JSON file and assign file name
URL = ["https://drive.google.com/uc?export=download&id=FILE_ID_CENTER_DYNAMIC",
       "https://drive.google.com/uc?export=download&id=FILE_ID_CENTER_FIX_1",
       "https://drive.google.com/uc?export=download&id=FILE_ID_CENTER_FIX_2",
       "https://drive.google.com/uc?export=download&id=FILE_ID_LEFT_DYNAMIC",
       "https://drive.google.com/uc?export=download&id=FILE_ID_LEFT_FIX_1",
       "https://drive.google.com/uc?export=download&id=FILE_ID_LEFT_FIX_2",
       "https://drive.google.com/uc?export=download&id=FILE_ID_RIGHT_DYNAMIC",
       "https://drive.google.com/uc?export=download&id=FILE_ID_RIGHT_FIX_1",
       "https://drive.google.com/uc?export=download&id=FILE_ID_RIGHT_FIX_2"]

fileName = ["Center_dynamic.json",
            "Center_fix_1.json",
            "Center_fix_2.json",
            "Left_dynamic.json",
            "Left_fix_1.json",
            "Left_fix_2.json",
            "Right_dynamic.json",
            "Right_fix_1.json",
            "Right_fix_2.json"]

for cnt in range(0,9):
  response = requests.get(URL[cnt])
  open(fileName[cnt], "wb").write(response.content)

In [ ]:
# Select Json File
for i in range(9):
  print(str(i) + ': ' + fileName[i])
selection = input('Select Json File by Entering Index 0~8 : ')
file_Name = fileName[int(selection)]
# open one of Json file
f = open(file_Name)

# returns JSON object as a dictionary
data = json.load(f)

# access all frame_urls
dic_slots = data['item']['slots']

# Iterating through the json list
# Access all annotation's frame number and x,y coordinates
list_annot = data['annotations']
list_nodes = []

for i in list_annot:
  dic_annot = i
  dic_annot_nodes = dic_annot['frames']
  dic_annot_frameNum = list(dic_annot_nodes.keys())
  dic_annot_nodes = dic_annot['frames'][dic_annot_frameNum[0]]['skeleton']['nodes']
  print('')
  print('frame_num: ', dic_annot_frameNum[0])
  list_nodes.append(int(dic_annot_frameNum[0]))

  #Iterating through all label
  for labelNum in range(0, 10):
    print('name of label: ',dic_annot_nodes[labelNum]['name'])
    print('x: ', dic_annot_nodes[labelNum]['x'])
    print('y: ', dic_annot_nodes[labelNum]['y'])

In [ ]:
import requests # request img from web
import shutil # save img locally
import os.path # save img in specific directory
import numpy as np
import pandas as pd
from google.colab.patches import cv2_imshow # for image display
from skimage import io
from PIL import Image
import matplotlib.pylab as plt

In [ ]:
# Make directory for download images
!mkdir images
save_path = 'images/'

In [ ]:
import cv2
from google.colab.patches import cv2_imshow
# Access specific url in frame_urls
nodes_num = 0
images_ALL = []


# Deal Image with Line and Circle
# circle and line color BGR
color = (0, 0, 255)
radius = 5
thickness = 2
line_color = (0, 0, 255)
line_thickness = 1
# To draw line, Rearrange label sequence in json file
line_Sequence = [0,1,2,4,7,9,6,3,5,8]


# for nodes_num in range(len(list_nodes)):
# for nodes_num in range(5):
for nodes_num in range(len(list_nodes)):
  x = []
  y = []
  frame_num = int(list_nodes[nodes_num])
  frame_URL = dic_slots[0]['frame_urls'][frame_num]
  # Download all images
  img_name = os.path.join(save_path, file_Name[:-5] + '_' + str(frame_num) + ".png")
  img_url = frame_URL
  res = requests.get(img_url, stream = True)
  if res.status_code == 200:
      with open(img_name,'wb') as f:
          shutil.copyfileobj(res.raw, f)
      print('Image sucessfully Downloaded: ',img_name)
  else:
      print('Image Could not be retrieved')

  #Resize and store cv2 image
  image = cv2.imread(img_name)
  image_Resized = cv2.resize(image, (320, 180))
  images_ALL.append(image_Resized)

  for label_cnt in range(10):
      x.append(list_annot[nodes_num]['frames'][str(list_nodes[nodes_num])]['skeleton']['nodes'][label_cnt]['x'])
      y.append(list_annot[nodes_num]['frames'][str(list_nodes[nodes_num])]['skeleton']['nodes'][label_cnt]['y'])
      # Center coordinates
      center_coordinates = (int(x[label_cnt]/4),int(y[label_cnt]/4))
      # Draw a circle
      if(label_cnt==0):
        image_Circling = cv2.circle(image_Resized, center_coordinates, radius, color, thickness)
      else:
        image_Circling = cv2.circle(image_Circling, center_coordinates, radius, color, thickness)

  for i in range(9):
      if (i<4):
        start = line_Sequence[i]
        end = line_Sequence[i+1]
      elif(i>=4 and i<7):
        start = line_Sequence[1]
        end = line_Sequence[i+1]
      else:
        start = line_Sequence[i]
        end = line_Sequence[i+1]
      start_point = (float(x[start])/4, float(y[start])/4)
      end_point = (float(x[end])/4, float(y[end])/4)
      image_Lined = cv2.line(image_Resized, (int(start_point[0]), int(start_point[1])), (int(end_point[0]), int(end_point[1])), line_color, line_thickness)

  # Displaying the image
  cv2_imshow(image_Lined)



# cv2_imshow(image_Resized)
# cv2.waitKey(0)

In [ ]:
# Closing file
print(len(list_nodes))
print(len(images_ALL))
f.close()